# OpenBB Tools

## Getting Started

### Install dependencies

In [ ]:
!pip install openbb
!pip install openbb-charting

!pip install pydantic==2.5

!pip install langchain
!pip install langchain_community
!pip install langchain_openai
# need to restart runtime after installing pydantic

### Set the OpenAI API Key

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = "your-api-key"

### Import packages

In [ ]:
from openbb import obb

### Login

In [ ]:
obb.account.login(pat="PAT")

## OpenBB tool pre-requisites

### Callable

In [ ]:
obb.equity.price.historical(symbol="AAPL", provider="polygon").to_df()

In [ ]:
obb.equity.price.historical(
    "AAPL", start_date="2022-01-01", provider="polygon"
).charting.to_chart()

### Input Schema

In [ ]:
obb.coverage.command_model[".equity.price.historical"]["openbb"]["QueryParams"]

In [ ]:
obb.coverage.command_model[".equity.price.historical"]["polygon"]["QueryParams"]

### Documentation

In [ ]:
help(obb.equity.price.historical)

## OpenBB tool

In [ ]:
from langchain_core.tools import StructuredTool

llm_historical_price = StructuredTool.from_function(
    func=obb.equity.price.historical,
    description=obb.equity.price.historical.__doc__.split("\n")[
        0
    ],  # Use first line of docstring
)

## Multiple OpenBB Tools

In [ ]:
llm_tools = [
    StructuredTool.from_function(
        name=name,
        func=schema["callable"],
        description=schema["callable"].__doc__.split("\n")[0],
    )
    for name, schema in obb.coverage.command_schemas().items()
]

## OpenBB Tools fed to agent

In [ ]:
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a very powerful assistant, but don't know current events"),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

llm_tools = [
    StructuredTool.from_function(
        func=obb.equity.price.quote,
        description=obb.equity.price.quote.__doc__.split("\n")[0],
    )
]

llm = ChatOpenAI(model="gpt-4", temperature=0.1)

agent = create_openai_functions_agent(llm=llm, tools=llm_tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=llm_tools, verbose=True)

agent_executor.invoke({"input": "What is the latest stock price of AAPL?"})